# Retrieval Experiments 01-99 Summary

이 노트북은 지금까지 쌓인 retrieval 실험 01-99번을 한눈에 보기 위해 만든 정리본입니다.

주의할 점은 실험군마다 기준이 완전히 같지 않다는 것입니다.

- 01-79번: 초기 500문항 baseline과 soyeon 125 데이터 실험이 섞여 있고, 일부는 예전 `hit@5` 산식으로 기록된 historical log입니다.
- 80-96번: 690 데이터 기준으로 다시 최적화한 실험입니다. 특히 96번은 다중 정답 문서에서 부분 점수를 주는 corrected partial hit 기준으로 500문항 전체를 재평가했습니다.
- 97-99번: 전체 실험 전 빠르게 확인한 screening입니다. 97/98은 206문항, 99는 100문항 기준이므로 96번 전체 점수와 직접 1:1 비교할 때는 scope를 같이 봐야 합니다.

요약 CSV도 함께 저장했습니다: `eval/evaluation/outputs/eval/experiment_01_99_summary.csv`


## 1. 평가 기준에서 바뀐 부분

이번 정리에서 가장 중요한 변화는 `hit@5` 해석입니다.

예전에는 다중 정답 문서 질문에서 정답 문서 2개 중 1개만 맞아도 hit가 1처럼 보일 수 있었습니다. 지금은 다중 정답 문서에서 1개만 맞추면 0.5, 2개 중 2개를 맞추면 1.0처럼 부분 점수로 봅니다.

그래서 96번 이후의 최종 판단에서는 `hit@5`, `doc_recall@5`, `multi_doc_recall@5`, `nDCG@5`를 같이 보는 것이 맞습니다.


## 2. 단계별 실험 흐름

| range | theme | summary |
| --- | --- | --- |
| 01-04 | 초기 baseline | BM25, dense, hybrid, keyword rerank를 500문항 canonical에서 처음 비교 |
| 05 | 새 데이터 파일 확인 | rag_database 계열 데이터로 baseline 가능성 확인 |
| 12-15 | DB/임베딩 비교 시작 | KoE5, BGE-M3, KURE 및 FAISS/Chroma 비교의 출발점 |
| 16-29 | diversity, hybrid, rerank, BM25 tokenizer | 문서 단위 평가에 맞추기 위해 후보 수와 중복 문서 억제를 실험 |
| 30-40 | 분석 지표와 document scoring | doc_recall, multi_doc_recall을 보고 chunk 검색을 문서 단위로 집계 |
| 41-50 | query decomposition/RRF/filter 완화 | 다중 문서 질문(type B)을 겨냥해 subquery와 RRF 조합을 실험 |
| 51-68 | parameter sweep | per-query candidates와 diversity candidates 조합 탐색 |
| 69-79 | 임베딩/Chroma 튜닝 | BGE-M3, KURE, Chroma metadata/HNSW 튜닝 비교. 125 기준 74번이 중요 후보 |
| 80-95 | 690 데이터 이전 및 최적화 | 125에서 찾은 구조를 690 데이터에 적용하고 doc scoring/target-aware를 보강 |
| 96 | 최종 best | partial hit 평가 기준에서 relaxed multi-agency filter + target-aware max5가 전체 500문항 최고 |
| 97-99 | 추가 실험 | hybrid, keyword rerank, cross-encoder rerank를 다시 확인했지만 96번보다 낮음 |

## 3. 중요 실험만 추린 표

아래 표는 전체 01-99 중에서 의사결정에 영향을 준 실험만 추린 것입니다.

| no | scope | setting_summary | hit_at_5 | mrr_at_5 | ndcg_at_5 | multi_doc_recall_at_5 | type_b_ndcg_at_5 | note |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | 초기 500문항 baseline | BM25 | 0.5760 | 0.5653 | 0.5681 |  |  | historical: old hit@5 기준일 수 있음 |
| 1 | 초기 500문항 baseline | BM25 canonical multi filter fix | 0.9760 | 0.9492 | 0.8302 |  |  | historical: old hit@5 기준일 수 있음 |
| 2 | 초기 500문항 baseline | dense openai small | 0.5840 | 0.5747 | 0.5771 |  |  | historical: old hit@5 기준일 수 있음 |
| 2 | 초기 500문항 baseline | dense openai small canonical multi filter fix | 0.9760 | 0.9501 | 0.8455 |  |  | historical: old hit@5 기준일 수 있음 |
| 3 | 초기 500문항 baseline | hybrid openai small | 0.5780 | 0.5650 | 0.5684 |  |  | historical: old hit@5 기준일 수 있음 |
| 3 | 초기 500문항 baseline | hybrid openai small canonical multi filter fix | 0.9780 | 0.9488 | 0.8300 |  |  | historical: old hit@5 기준일 수 있음 |
| 4 | 초기 500문항 baseline | hybrid openai small rerank | 0.5860 | 0.5763 | 0.5788 |  |  | historical: old hit@5 기준일 수 있음 |
| 4 | 초기 500문항 baseline | hybrid openai small rerank canonical multi filter fix | 0.9860 | 0.9649 | 0.8399 |  |  | historical: old hit@5 기준일 수 있음 |
| 54 | soyeon 125 실험 | dense conditional query decomposition v2 RRF multi relaxed filter kept per50 document diversification100 soyeon 125 KoE5 | 1.0000 | 0.9700 | 0.9486 |  |  | historical: old hit@5 기준일 수 있음 |
| 69 | soyeon 125 실험 | dense conditional query decomposition v2 RRF multi relaxed filter kept per50 document diversification100 soyeon 125 BGE- | 1.0000 | 0.9618 | 0.9399 |  |  | historical: old hit@5 기준일 수 있음 |
| 70 | soyeon 125 실험 | dense conditional query decomposition v2 RRF multi relaxed filter kept per50 document diversification100 soyeon 125 KURE | 1.0000 | 0.9690 | 0.9525 |  |  | historical: old hit@5 기준일 수 있음 |
| 71 | soyeon 125 실험 | dense conditional query decomposition v2 RRF multi relaxed filter kept per50 document diversification200 soyeon 125 KURE | 1.0000 | 0.9690 | 0.9548 |  |  | historical: old hit@5 기준일 수 있음 |
| 74 | soyeon 125 실험 | dense conditional query decomposition v2 RRF multi relaxed filter kept per50 document diversification250 soyeon 125 KURE | 1.0000 | 0.9738 | 0.9594 |  |  | historical: old hit@5 기준일 수 있음 |
| 80 | 690 full / partial hit 재평가 | dense query decomposition RRF document diversification250 KURE Chroma 690 | 0.9370 | 0.9390 | 0.9068 | 0.8448 | 0.8292 | corrected partial hit 기준 |
| 81 | 690 full / partial hit 재평가 | dense query decomposition RRF document diversification250 keyword rerank KURE Chroma 690 | 0.8718 | 0.9358 | 0.8606 | 0.6892 | 0.7194 | corrected partial hit 기준 |
| 91 | 690 full / partial hit 재평가 | dense query decomposition RRF per75 document scoring mean3 300 KURE Chroma 690 | 0.9375 | 0.9473 | 0.9113 | 0.8461 | 0.8378 | corrected partial hit 기준 |
| 95 | 690 full / partial hit 재평가 | dense query decomposition RRF per75 document scoring mean3 target-aware30 preserve3 KURE Chroma 690 | 0.9388 | 0.9477 | 0.9119 | 0.8493 | 0.8395 | corrected partial hit 기준 |
| 96 | 690 full / partial hit 재평가 | dense query decomposition RRF per75 document scoring mean3 target-aware30 max5 preserve3 relaxed filter KURE Chroma 690 | 0.9640 | 0.9467 | 0.9255 | 0.9113 | 0.8734 | corrected partial hit 기준 |
| 97 | 690 partial 206 screening | hybrid partial screening | 0.9288 | 0.9330 | 0.9040 | 0.8417 | 0.8146 | same 206 questions screening |
| 98 | 690 partial 206 screening | keyword rerank partial screening | 0.9066 | 0.8876 | 0.8553 | 0.7719 | 0.7153 | same 206 questions screening |
| 99 | 690 Q001-Q100 screening | 96 final dense setting + BAAI/bge-reranker-v2-m3 cross-encoder rerank | 0.9650 | 0.8852 | 0.8906 | 0.9125 | 0.8308 | same 100 questions; 96 same100 nDCG=0.9414보다 낮음 |

## 4. 690 데이터 기준 상위 실험

아래 표는 690 데이터에서 corrected partial hit 기준으로 다시 평가한 full 실험 중 nDCG 순위가 높은 실험입니다.

| no | setting_summary | hit_at_5 | mrr_at_5 | ndcg_at_5 | multi_doc_recall_at_5 | type_b_ndcg_at_5 |
| --- | --- | --- | --- | --- | --- | --- |
| 96 | dense query decomposition RRF per75 document scoring mean3 target-aware30 max5 preserve3 relaxed filter KURE Chroma 690 | 0.9640 | 0.9467 | 0.9255 | 0.9113 | 0.8734 |
| 95 | dense query decomposition RRF per75 document scoring mean3 target-aware30 preserve3 KURE Chroma 690 | 0.9388 | 0.9477 | 0.9119 | 0.8493 | 0.8395 |
| 91 | dense query decomposition RRF per75 document scoring mean3 300 KURE Chroma 690 | 0.9375 | 0.9473 | 0.9113 | 0.8461 | 0.8378 |
| 93 | dense query decomposition RRF per75 document scoring sum3 300 KURE Chroma 690 | 0.9372 | 0.9473 | 0.9113 | 0.8452 | 0.8378 |
| 87 | dense query decomposition RRF per75 document diversification300 KURE Chroma 690 | 0.9392 | 0.9391 | 0.9079 | 0.8502 | 0.8322 |
| 92 | dense query decomposition RRF per75 document scoring max 300 KURE Chroma 690 | 0.9392 | 0.9391 | 0.9079 | 0.8502 | 0.8322 |
| 88 | dense query decomposition RRF per75 document diversification400 KURE Chroma 690 | 0.9392 | 0.9391 | 0.9079 | 0.8502 | 0.8322 |
| 86 | dense query decomposition RRF per75 document diversification250 KURE Chroma 690 | 0.9385 | 0.9391 | 0.9076 | 0.8485 | 0.8313 |
| 84 | dense query decomposition RRF per50 document diversification400 KURE Chroma 690 | 0.9377 | 0.9390 | 0.9071 | 0.8465 | 0.8301 |
| 83 | dense query decomposition RRF per50 document diversification300 KURE Chroma 690 | 0.9377 | 0.9390 | 0.9071 | 0.8465 | 0.8301 |

## 5. 전체 실험 목록

긴 표이지만, 01-99까지 어떤 실험이 어떤 역할을 했는지 남겨두기 위한 기록입니다. `historical`이라고 적힌 실험은 현재 최종 평가 기준과 직접 비교하기보다는 당시 의사결정 흐름을 보는 용도로 사용합니다.

| no | scope | setting_summary | hit_at_5 | mrr_at_5 | ndcg_at_5 | note |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | 초기 500문항 baseline | BM25 | 0.5760 | 0.5653 | 0.5681 | historical: old hit@5 기준일 수 있음 |
| 1 | 초기 500문항 baseline | BM25 canonical multi filter fix | 0.9760 | 0.9492 | 0.8302 | historical: old hit@5 기준일 수 있음 |
| 2 | 초기 500문항 baseline | dense openai small | 0.5840 | 0.5747 | 0.5771 | historical: old hit@5 기준일 수 있음 |
| 2 | 초기 500문항 baseline | dense openai small canonical multi filter fix | 0.9760 | 0.9501 | 0.8455 | historical: old hit@5 기준일 수 있음 |
| 3 | 초기 500문항 baseline | hybrid openai small | 0.5780 | 0.5650 | 0.5684 | historical: old hit@5 기준일 수 있음 |
| 3 | 초기 500문항 baseline | hybrid openai small canonical multi filter fix | 0.9780 | 0.9488 | 0.8300 | historical: old hit@5 기준일 수 있음 |
| 4 | 초기 500문항 baseline | hybrid openai small rerank | 0.5860 | 0.5763 | 0.5788 | historical: old hit@5 기준일 수 있음 |
| 4 | 초기 500문항 baseline | hybrid openai small rerank canonical multi filter fix | 0.9860 | 0.9649 | 0.8399 | historical: old hit@5 기준일 수 있음 |
| 5 | 초기 500문항 baseline | BM25 rag database | 0.8060 | 0.7387 | 0.6527 | historical: old hit@5 기준일 수 있음 |
| 12 | soyeon 125 실험 | hybrid rerank soyeon 125 KoE5 FAISS | 0.9880 | 0.9593 | 0.8471 | historical: old hit@5 기준일 수 있음 |
| 13 | soyeon 125 실험 | hybrid rerank soyeon 125 KoE5 Chroma | 0.9880 | 0.9593 | 0.8471 | historical: old hit@5 기준일 수 있음 |
| 14 | soyeon 125 실험 | hybrid rerank soyeon 125 BGE-M3 FAISS | 0.9880 | 0.9592 | 0.8467 | historical: old hit@5 기준일 수 있음 |
| 15 | soyeon 125 실험 | hybrid rerank soyeon 125 KURE FAISS | 0.9880 | 0.9592 | 0.8470 | historical: old hit@5 기준일 수 있음 |
| 16 | soyeon 125 실험 | hybrid rerank document diversification fetch100 soyeon 125 KoE5 FAISS | 0.9940 | 0.9622 | 0.9201 | historical: old hit@5 기준일 수 있음 |
| 17 | soyeon 125 실험 | hybrid rerank document diversification fetch100 BM25 07 dense 03 soyeon 125 KoE5 FAISS | 0.9940 | 0.9642 | 0.9213 | historical: old hit@5 기준일 수 있음 |
| 18 | soyeon 125 실험 | hybrid rerank document diversification fetch100 BM25 03 dense 07 soyeon 125 KoE5 FAISS | 0.9940 | 0.9582 | 0.9177 | historical: old hit@5 기준일 수 있음 |
| 19 | soyeon 125 실험 | dense rerank document diversification100 soyeon 125 KoE5 FAISS | 0.9980 | 0.9585 | 0.9238 | historical: old hit@5 기준일 수 있음 |
| 20 | soyeon 125 실험 | dense document diversification100 no rerank soyeon 125 KoE5 FAISS | 0.9980 | 0.9708 | 0.9286 | historical: old hit@5 기준일 수 있음 |
| 21 | soyeon 125 실험 | dense rerank weight010 document diversification100 soyeon 125 KoE5 FAISS | 0.9980 | 0.9575 | 0.9235 | historical: old hit@5 기준일 수 있음 |
| 22 | soyeon 125 실험 | dense document diversification100 no rerank docnorm soyeon 125 KoE5 FAISS | 1.0000 | 0.9728 | 0.9306 | historical: old hit@5 기준일 수 있음 |
| 23 | soyeon 125 실험 | dense document diversification100 soyeon 125 BGE-M3 FAISS | 1.0000 | 0.9595 | 0.9164 | historical: old hit@5 기준일 수 있음 |
| 24 | soyeon 125 실험 | dense document diversification100 soyeon 125 KURE FAISS | 1.0000 | 0.9696 | 0.9259 | historical: old hit@5 기준일 수 있음 |
| 25 | soyeon 125 실험 | dense cross encoder rerank10 document diversification10 soyeon 125 KoE5 FAISS | 1.0000 | 0.9648 | 0.9041 | historical: old hit@5 기준일 수 있음 |
| 26 | soyeon 125 실험 | dense document diversification100 cross encoder rerank10 soyeon 125 KoE5 FAISS | 0.9840 | 0.9643 | 0.8652 | historical: old hit@5 기준일 수 있음 |
| 27 | soyeon 125 실험 | dense keyword rerank document diversification100 docnorm soyeon 125 KoE5 FAISS | 1.0000 | 0.9605 | 0.9258 | historical: old hit@5 기준일 수 있음 |
| 28 | soyeon 125 실험 | hybrid whitespace BM25 dense03 document diversification100 soyeon 125 KoE5 FAISS | 0.9920 | 0.9493 | 0.9100 | historical: old hit@5 기준일 수 있음 |
| 29 | soyeon 125 실험 | hybrid regex BM25 dense03 document diversification100 soyeon 125 KoE5 FAISS | 0.9960 | 0.9587 | 0.9199 | historical: old hit@5 기준일 수 있음 |
| 30 | soyeon 125 실험 | analysis dense best doc recall soyeon 125 KoE5 FAISS | 1.0000 | 0.9728 | 0.9306 | historical: old hit@5 기준일 수 있음 |
| 31 | soyeon 125 실험 | analysis hybrid whitespace doc recall soyeon 125 KoE5 FAISS | 0.9920 | 0.9493 | 0.9100 | historical: old hit@5 기준일 수 있음 |
| 32 | soyeon 125 실험 | analysis hybrid regex doc recall soyeon 125 KoE5 FAISS | 0.9960 | 0.9587 | 0.9199 | historical: old hit@5 기준일 수 있음 |
| 33 | soyeon 125 실험 | dense document scoring max100 soyeon 125 KoE5 FAISS | 1.0000 | 0.9728 | 0.9306 | historical: old hit@5 기준일 수 있음 |
| 34 | soyeon 125 실험 | dense document scoring mean top3 100 soyeon 125 KoE5 FAISS | 1.0000 | 0.9608 | 0.9253 | historical: old hit@5 기준일 수 있음 |
| 35 | soyeon 125 실험 | dense document scoring sum top3 100 soyeon 125 KoE5 FAISS | 1.0000 | 0.9618 | 0.9274 | historical: old hit@5 기준일 수 있음 |
| 36 | soyeon 125 실험 | dense document diversification10 soyeon 125 KoE5 FAISS | 1.0000 | 0.9728 | 0.9072 | historical: old hit@5 기준일 수 있음 |
| 37 | soyeon 125 실험 | dense document diversification20 soyeon 125 KoE5 FAISS | 1.0000 | 0.9728 | 0.9194 | historical: old hit@5 기준일 수 있음 |
| 38 | soyeon 125 실험 | dense document diversification50 soyeon 125 KoE5 FAISS | 1.0000 | 0.9728 | 0.9270 | historical: old hit@5 기준일 수 있음 |
| 39 | soyeon 125 실험 | dense document diversification200 soyeon 125 KoE5 FAISS | 1.0000 | 0.9728 | 0.9316 | historical: old hit@5 기준일 수 있음 |
| 40 | soyeon 125 실험 | dense document diversification200 no multi filter soyeon 125 KoE5 FAISS | 0.9920 | 0.9537 | 0.9239 | historical: old hit@5 기준일 수 있음 |
| 41 | soyeon 125 실험 | dense query decomposition roundrobin filter document diversification200 soyeon 125 KoE5 FAISS | 1.0000 | 0.9637 | 0.9282 | historical: old hit@5 기준일 수 있음 |
| 42 | soyeon 125 실험 | dense query decomposition roundrobin relaxed document diversification200 soyeon 125 KoE5 FAISS | 0.9780 | 0.9162 | 0.9033 | historical: old hit@5 기준일 수 있음 |
| 43 | soyeon 125 실험 | dense query decomposition RRF relaxed per100 document diversification200 soyeon 125 KoE5 FAISS | 0.9840 | 0.9273 | 0.9199 | historical: old hit@5 기준일 수 있음 |
| 44 | soyeon 125 실험 | dense query decomposition RRF filter per100 document diversification200 soyeon 125 KoE5 FAISS | 1.0000 | 0.9548 | 0.9269 | historical: old hit@5 기준일 수 있음 |
| 45 | soyeon 125 실험 | dense query decomposition RRF filter per200 document diversification200 soyeon 125 KoE5 FAISS | 1.0000 | 0.9512 | 0.9247 | historical: old hit@5 기준일 수 있음 |
| 46 | soyeon 125 실험 | dense conditional query decomposition RRF filter per100 document diversification200 soyeon 125  | 1.0000 | 0.9702 | 0.9355 | historical: old hit@5 기준일 수 있음 |
| 47 | soyeon 125 실험 | dense conditional query decomposition RRF relaxed per100 document diversification200 soyeon 125 | 0.9980 | 0.9543 | 0.9393 | historical: old hit@5 기준일 수 있음 |
| 48 | soyeon 125 실험 | dense conditional query decomposition v2 RRF filter per100 document diversification200 soyeon 1 | 1.0000 | 0.9738 | 0.9367 | historical: old hit@5 기준일 수 있음 |
| 49 | soyeon 125 실험 | dense conditional query decomposition v2 RRF relaxed per100 document diversification200 soyeon  | 1.0000 | 0.9660 | 0.9456 | historical: old hit@5 기준일 수 있음 |
| 50 | soyeon 125 실험 | dense conditional query decomposition v2 RRF multi relaxed filter kept per100 document diversif | 1.0000 | 0.9690 | 0.9478 | historical: old hit@5 기준일 수 있음 |
| 51 | soyeon 125 실험 | dense conditional query decomposition v2 RRF multi relaxed filter kept per100 document diversif | 1.0000 | 0.9690 | 0.9478 | historical: old hit@5 기준일 수 있음 |
| 52 | soyeon 125 실험 | dense conditional query decomposition v2 RRF multi relaxed filter kept per50 document diversifi | 1.0000 | 0.9700 | 0.9472 | historical: old hit@5 기준일 수 있음 |
| 53 | soyeon 125 실험 | dense conditional query decomposition v2 RRF multi relaxed filter kept per50 document diversifi | 1.0000 | 0.9700 | 0.9480 | historical: old hit@5 기준일 수 있음 |
| 54 | soyeon 125 실험 | dense conditional query decomposition v2 RRF multi relaxed filter kept per50 document diversifi | 1.0000 | 0.9700 | 0.9486 | historical: old hit@5 기준일 수 있음 |
| 55 | soyeon 125 실험 | dense conditional query decomposition v2 RRF multi relaxed filter kept per75 document diversifi | 1.0000 | 0.9690 | 0.9462 | historical: old hit@5 기준일 수 있음 |
| 56 | soyeon 125 실험 | dense conditional query decomposition v2 RRF multi relaxed filter kept per75 document diversifi | 1.0000 | 0.9690 | 0.9470 | historical: old hit@5 기준일 수 있음 |
| 57 | soyeon 125 실험 | dense conditional query decomposition v2 RRF multi relaxed filter kept per75 document diversifi | 1.0000 | 0.9690 | 0.9476 | historical: old hit@5 기준일 수 있음 |
| 58 | soyeon 125 실험 | dense conditional query decomposition v2 RRF multi relaxed filter kept per100 document diversif | 1.0000 | 0.9690 | 0.9465 | historical: old hit@5 기준일 수 있음 |
| 59 | soyeon 125 실험 | dense conditional query decomposition v2 RRF multi relaxed filter kept per100 document diversif | 1.0000 | 0.9690 | 0.9472 | historical: old hit@5 기준일 수 있음 |
| 60 | soyeon 125 실험 | dense conditional query decomposition v2 RRF multi relaxed filter kept per150 document diversif | 1.0000 | 0.9660 | 0.9450 | historical: old hit@5 기준일 수 있음 |
| 61 | soyeon 125 실험 | dense conditional query decomposition v2 RRF multi relaxed filter kept per150 document diversif | 1.0000 | 0.9660 | 0.9458 | historical: old hit@5 기준일 수 있음 |
| 62 | soyeon 125 실험 | dense conditional query decomposition v2 RRF multi relaxed filter kept per150 document diversif | 1.0000 | 0.9660 | 0.9464 | historical: old hit@5 기준일 수 있음 |
| 63 | soyeon 125 실험 | dense conditional query decomposition v2 RRF multi relaxed filter kept per25 document diversifi | 1.0000 | 0.9710 | 0.9472 | historical: old hit@5 기준일 수 있음 |
| 64 | soyeon 125 실험 | dense conditional query decomposition v2 RRF multi relaxed filter kept per25 document diversifi | 1.0000 | 0.9710 | 0.9478 | historical: old hit@5 기준일 수 있음 |
| 65 | soyeon 125 실험 | dense conditional query decomposition v2 RRF multi relaxed filter kept per40 document diversifi | 1.0000 | 0.9700 | 0.9475 | historical: old hit@5 기준일 수 있음 |
| 66 | soyeon 125 실험 | dense conditional query decomposition v2 RRF multi relaxed filter kept per40 document diversifi | 1.0000 | 0.9700 | 0.9481 | historical: old hit@5 기준일 수 있음 |
| 67 | soyeon 125 실험 | dense conditional query decomposition v2 RRF multi relaxed filter kept per50 document diversifi | 1.0000 | 0.9700 | 0.9486 | historical: old hit@5 기준일 수 있음 |
| 68 | soyeon 125 실험 | dense conditional query decomposition v2 RRF multi relaxed filter kept per50 document diversifi | 1.0000 | 0.9700 | 0.9486 | historical: old hit@5 기준일 수 있음 |
| 69 | soyeon 125 실험 | dense conditional query decomposition v2 RRF multi relaxed filter kept per50 document diversifi | 1.0000 | 0.9618 | 0.9399 | historical: old hit@5 기준일 수 있음 |
| 70 | soyeon 125 실험 | dense conditional query decomposition v2 RRF multi relaxed filter kept per50 document diversifi | 1.0000 | 0.9690 | 0.9525 | historical: old hit@5 기준일 수 있음 |
| 71 | soyeon 125 실험 | dense conditional query decomposition v2 RRF multi relaxed filter kept per50 document diversifi | 1.0000 | 0.9690 | 0.9548 | historical: old hit@5 기준일 수 있음 |
| 72 | soyeon 125 실험 | dense conditional query decomposition v2 RRF multi relaxed filter kept per50 document diversifi | 1.0000 | 0.9718 | 0.9523 | historical: old hit@5 기준일 수 있음 |
| 73 | soyeon 125 실험 | dense conditional query decomposition v2 RRF multi relaxed filter kept per50 document diversifi | 1.0000 | 0.9738 | 0.9584 | historical: old hit@5 기준일 수 있음 |
| 74 | soyeon 125 실험 | dense conditional query decomposition v2 RRF multi relaxed filter kept per50 document diversifi | 1.0000 | 0.9738 | 0.9594 | historical: old hit@5 기준일 수 있음 |
| 75 | soyeon 125 실험 | dense conditional query decomposition v2 RRF multi relaxed filter kept per50 document diversifi | 1.0000 | 0.9738 | 0.9594 | historical: old hit@5 기준일 수 있음 |
| 76 | soyeon 125 실험 | dense conditional query decomposition v2 RRF multi relaxed filter kept per75 document diversifi | 1.0000 | 0.9715 | 0.9569 | historical: old hit@5 기준일 수 있음 |
| 77 | soyeon 125 실험 | dense conditional query decomposition v2 RRF multi relaxed filter kept per75 document diversifi | 1.0000 | 0.9715 | 0.9579 | historical: old hit@5 기준일 수 있음 |
| 78 | soyeon 125 실험 | dense conditional query decomposition v2 RRF multi relaxed filter kept per50 document diversifi | 1.0000 | 0.9738 | 0.9584 | historical: old hit@5 기준일 수 있음 |
| 79 | soyeon 125 실험 | dense conditional query decomposition v2 RRF multi relaxed filter kept per50 document diversifi | 1.0000 | 0.9738 | 0.9584 | historical: old hit@5 기준일 수 있음 |
| 80 | 690 full / partial hit 재평가 | dense query decomposition RRF document diversification250 KURE Chroma 690 | 0.9370 | 0.9390 | 0.9068 | corrected partial hit 기준 |
| 81 | 690 full / partial hit 재평가 | dense query decomposition RRF document diversification250 keyword rerank KURE Chroma 690 | 0.8718 | 0.9358 | 0.8606 | corrected partial hit 기준 |
| 82 | 690 full / partial hit 재평가 | dense query decomposition RRF per50 document diversification200 KURE Chroma 690 | 0.9353 | 0.9390 | 0.9055 | corrected partial hit 기준 |
| 83 | 690 full / partial hit 재평가 | dense query decomposition RRF per50 document diversification300 KURE Chroma 690 | 0.9377 | 0.9390 | 0.9071 | corrected partial hit 기준 |
| 84 | 690 full / partial hit 재평가 | dense query decomposition RRF per50 document diversification400 KURE Chroma 690 | 0.9377 | 0.9390 | 0.9071 | corrected partial hit 기준 |
| 85 | 690 full / partial hit 재평가 | dense query decomposition RRF per50 document diversification250 source file KURE Chroma 690 | 0.9370 | 0.9390 | 0.9068 | corrected partial hit 기준 |
| 86 | 690 full / partial hit 재평가 | dense query decomposition RRF per75 document diversification250 KURE Chroma 690 | 0.9385 | 0.9391 | 0.9076 | corrected partial hit 기준 |
| 87 | 690 full / partial hit 재평가 | dense query decomposition RRF per75 document diversification300 KURE Chroma 690 | 0.9392 | 0.9391 | 0.9079 | corrected partial hit 기준 |
| 88 | 690 full / partial hit 재평가 | dense query decomposition RRF per75 document diversification400 KURE Chroma 690 | 0.9392 | 0.9391 | 0.9079 | corrected partial hit 기준 |
| 89 | 690 full / partial hit 재평가 | dense query decomposition RRF per100 document diversification300 KURE Chroma 690 | 0.9395 | 0.9368 | 0.9070 | corrected partial hit 기준 |
| 90 | 690 full / partial hit 재평가 | dense query decomposition RRF per100 document diversification400 KURE Chroma 690 | 0.9395 | 0.9368 | 0.9070 | corrected partial hit 기준 |
| 91 | 690 full / partial hit 재평가 | dense query decomposition RRF per75 document scoring mean3 300 KURE Chroma 690 | 0.9375 | 0.9473 | 0.9113 | corrected partial hit 기준 |
| 92 | 690 full / partial hit 재평가 | dense query decomposition RRF per75 document scoring max 300 KURE Chroma 690 | 0.9392 | 0.9391 | 0.9079 | corrected partial hit 기준 |
| 93 | 690 full / partial hit 재평가 | dense query decomposition RRF per75 document scoring sum3 300 KURE Chroma 690 | 0.9372 | 0.9473 | 0.9113 | corrected partial hit 기준 |
| 94 | 690 full / partial hit 재평가 | dense query decomposition RRF per75 document scoring mean3 target-aware30 max3 KURE Chroma 690 | 0.9122 | 0.8837 | 0.8557 | corrected partial hit 기준 |
| 95 | 690 full / partial hit 재평가 | dense query decomposition RRF per75 document scoring mean3 target-aware30 preserve3 KURE Chroma | 0.9388 | 0.9477 | 0.9119 | corrected partial hit 기준 |
| 96 | 690 full / partial hit 재평가 | dense query decomposition RRF per75 document scoring mean3 target-aware30 max5 preserve3 relaxe | 0.9640 | 0.9467 | 0.9255 | corrected partial hit 기준 |
| 97 | 690 partial 206 screening | hybrid partial screening | 0.9288 | 0.9330 | 0.9040 | same 206 questions screening |
| 98 | 690 partial 206 screening | keyword rerank partial screening | 0.9066 | 0.8876 | 0.8553 | same 206 questions screening |
| 99 | 690 Q001-Q100 screening | 96 final dense setting + BAAI/bge-reranker-v2-m3 cross-encoder rerank | 0.9650 | 0.8852 | 0.8906 | same 100 questions; 96 same100 nDCG=0.9414보다 낮음 |

## 6. 최종 결론

최종적으로 가장 성능이 좋았던 조합은 96번입니다.

- retriever: dense
- embedding: KURE
- vector store: Chroma
- query decomposition: conditional + RRF
- document scoring: mean_top_n, top 3 chunk 평균
- target-aware: 사용
- target max count: 5
- target base preserve: 3
- multi-agency filter: relaxed

96번 전체 500문항 결과는 다음과 같습니다.

| metric | score |
|---|---:|
| hit@5 | 0.9640 |
| MRR@5 | 0.9467 |
| nDCG@5 | 0.9255 |
| doc recall@5 | 0.9640 |
| multi-doc recall@5 | 0.9113 |
| type B nDCG@5 | 0.8734 |

hybrid, keyword rerank, cross-encoder rerank는 다시 확인했지만 지금 구조에서는 96번보다 낮았습니다. 특히 generation으로 넘길 때는 여러 정답 문서를 빠뜨리지 않는 것이 중요하므로, 현재는 96번 결과를 최종 retrieval 입력으로 쓰는 것이 가장 합리적입니다.


## 7. 정리 후 남겨둘 prediction 파일

용량 정리를 위해 중간 prediction JSONL은 삭제하고, 아래 파일만 보존하는 기준으로 정리했습니다.

- `54_dense_conditional_qdecomp_v2_rrf_multi_relaxed_filter_kept_per50_diverse100_soyeon_125_koe5_faiss_canonical.jsonl`
- `69_dense_conditional_qdecomp_v2_rrf_multi_relaxed_filter_kept_per50_diverse100_soyeon_125_bge_m3_faiss_canonical.jsonl`
- `70_dense_conditional_qdecomp_v2_rrf_multi_relaxed_filter_kept_per50_diverse100_soyeon_125_kure_faiss_canonical.jsonl`
- `71_dense_conditional_qdecomp_v2_rrf_multi_relaxed_filter_kept_per50_diverse200_soyeon_125_kure_faiss_canonical.jsonl`
- `74_dense_conditional_qdecomp_v2_rrf_multi_relaxed_filter_kept_per50_diverse250_soyeon_125_kure_chroma_hnsw_tuned_canonical.jsonl`
- `80_dense_qdecomp_rrf_diverse250_kure_chroma_690_canonical.jsonl`
- `91_dense_qdecomp_rrf_per75_docscore_mean3_300_kure_chroma_690_canonical.jsonl`
- `95_dense_qdecomp_rrf_per75_docscore_mean3_targetaware30_preserve3_kure_chroma_690_canonical.jsonl`
- `96_dense_qdecomp_rrf_per75_docscore_mean3_targetaware30_max5_preserve3_relaxed_filter_kure_chroma_690_canonical.jsonl`
- `97_hybrid_bm25_03_dense_07_qdecomp_rrf_per75_docscore_mean3_targetaware30_max5_preserve3_relaxed_filter_kure_chroma_690_partial206.jsonl`
- `98_dense_relaxed_targetaware_keyword_rerank_after_limit206_kure_chroma_690_canonical.jsonl`
- `99_dense_final_crossencoder_after_limit100_kure_chroma_690_canonical.jsonl`

나머지는 재현성이 아주 중요한 소스 코드나 평가 로그가 아니라, 중간 실험의 생성 결과물이므로 삭제 대상으로 봤습니다. 점수와 핵심 해석은 이 노트북과 `experiment_01_99_summary.csv`에 남겨두었습니다.
